# 01 — PEARL-Lite: HC3 → GPT3.5-Mixed 0% và HC3 Insert 25%

Theo luồng notebook LISA: **clone → train HC3 → test → ZIP có seed**.
Bật **Internet + GPU** trên Kaggle. Push code lên `version4` trước khi Run All.
Notebook gọi các module trong repo, không nhúng snapshot mã nguồn.

Mặc định chạy toàn bộ dữ liệu với **seed 2026**. Chọn `SMOKE_TEST=True` để chạy 24 train/8 validation
và 4 mẫu/điều kiện. Hai tập test không tham gia chọn checkpoint; threshold cố định 0.5.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, time, hashlib, zipfile

SEED = 2026
BASELINE_EPOCHS = 40
EDGE_EPOCHS = 20
EPOCHS = 10  # PEARL fine-tune
PATIENCE = 5  # Baseline validation accuracy
EPSILON, LAMBDA_EDGE = 0.8, 0.1
BRANCH = 'version4'
REPO_URL = 'https://github.com/tphan1420/PRDetect-Research.git'
SMOKE_TEST = False
WORK = Path('/kaggle/working/pearl_hc3_gpt35_0pct_insert25')

# Optional local verification hooks; trên Kaggle không cần đặt các biến này.
SMOKE_TEST = os.environ.get('PEARL_NOTEBOOK_SMOKE', str(int(SMOKE_TEST))) == '1'
WORK = Path(os.environ.get('PEARL_NOTEBOOK_WORK', str(WORK))).resolve()
REPO = Path(os.environ.get('PEARL_EXISTING_REPO', str(WORK / 'PRDetect-Research'))).resolve()
MODEL_DIR, EVAL_DIR = WORK / f'seed{SEED}/model', WORK / f'seed{SEED}/evaluation'
RUN = WORK / f'seed{SEED}'
WORK.mkdir(parents=True, exist_ok=True)
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HOME'] = str(WORK / 'hf_cache')
print({'seed': SEED, 'branch': BRANCH, 'smoke': SMOKE_TEST, 'work': str(WORK)})


In [ ]:
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO)], check=True)
assert (REPO / 'pearl/train_gcn.py').is_file(), 'Hãy push code PEARL lên version4 trước.'
COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
assert subprocess.check_output(['git', '-C', str(REPO), 'branch', '--show-current'], text=True).strip() == BRANCH
if os.environ.get('PEARL_SKIP_INSTALL') != '1':
    # Giữ nguyên torch CUDA của Kaggle.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-geometric>=2.5,<3',
        'transformers>=4.48,<5', 'spacy==3.8.7', 'click>=8', 'scikit-learn>=1.3'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
        'https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl'], check=True)
sys.path.insert(0, str(REPO))
os.chdir(REPO)
import torch, spacy
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print({'commit': COMMIT, 'device': DEVICE})

def run(stage, arguments):
    logs = RUN / 'logs'
    logs.mkdir(parents=True, exist_ok=True)
    command = [sys.executable, '-u', *map(str, arguments)]
    env = dict(os.environ, PYTHONHASHSEED=str(SEED), PYTHONUNBUFFERED='1')
    with (logs / f'{stage}.log').open('w', encoding='utf-8') as log:
        process = subprocess.Popen(command, cwd=REPO, env=env, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace')
        try:
            for line in process.stdout:
                log.write(line)
                print(line, end='', flush=True)
            status = process.wait()
        except BaseException:
            process.terminate()
            process.wait()
            raise
    if status:
        raise subprocess.CalledProcessError(status, command)


## Chuẩn bị HC3 và retention pairs
GCN warm-start được train bằng script gốc, trên cùng HC3 derivative đã quarantine overlap
với validation. Không cần upload checkpoint baseline. Synonym/replace/insert/repeat/delete tạo
nhãn từ provenance alignment; mất endpoint hoặc split/merge là unknown, không phải negative.


In [ ]:
TRAIN_JSON = REPO / 'data/original_text/benchmarks/hc3/hc3_train.json'
VAL_JSON = REPO / 'data/original_text/benchmarks/hc3/hc3_val.json'
assert TRAIN_JSON.is_file() and VAL_JSON.is_file()
CONFIG = RUN / 'config.json'
assert not CONFIG.exists(), 'Run đã tồn tại. Đổi WORK để tránh ghi đè.'
RUN.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)
EVAL_DIR.mkdir(exist_ok=True)
ROBERTA = Path(os.environ.get('PEARL_ROBERTA_LOCAL', str(REPO / 'roberta-base'))).resolve()
if not (ROBERTA / 'config.json').is_file():
    run('download_roberta', ['-m', 'pearl.download_roberta', '--output', ROBERTA])
BASELINE = MODEL_DIR / f'hc3_gcn_seed{SEED}.pth'
cfg = json.loads((REPO / 'configs/pearl_hc3.json').read_text(encoding='utf-8'))
cfg.update(seed=SEED, device=DEVICE, source_train=str(TRAIN_JSON), source_validation=str(VAL_JSON),
    baseline_checkpoint=str(BASELINE), roberta=str(ROBERTA), local_files_only=True,
    prepared=str(RUN / 'prepared'), edge_output=str(MODEL_DIR / 'edge'), gcn_output=str(MODEL_DIR / 'pearl'),
    edge_epochs=3 if SMOKE_TEST else EDGE_EPOCHS, gcn_epochs=2 if SMOKE_TEST else EPOCHS,
    epsilon=EPSILON, lambda_edge=LAMBDA_EDGE, notebook_commit=COMMIT,
    train_limit=24 if SMOKE_TEST else None, validation_limit=8 if SMOKE_TEST else None,
    data_root=str(REPO / 'data'), shuffle_control=False,
    evaluation_entries=[
        {'table': 'table3', 'dataset': 'gpt35_mixed', 'condition': 'original',
         'path': str(REPO / 'data/original_text/benchmarks/gpt35_mixed/gpt3.5_mixed_test_split.json')},
        {'table': 'table4', 'dataset': 'hc3', 'condition': 'insert25',
         'path': str(REPO / 'data/perturbed_text/hc3/structural_noise/hc3_test_insert_25.json')}])
CONFIG.write_text(json.dumps(cfg, indent=2), encoding='utf-8')
run('prepare', ['-m', 'pearl.prepare', '--config', CONFIG])


In [ ]:
# Build đúng 2 graph caches sạch cho script GCN gốc, rồi xóa sau warm-start.
# Gọi graph builder gốc với cùng parser/RoBERTa; không sửa source baseline.
import gc
import building_graph as builder
from pearl.data import read_jsonl
from pearl.config import seed_all
from pearl.embeddings import LegacyEncoder
seed_all(SEED)
encoder = LegacyEncoder(cfg)
builder.nlp, builder.model = spacy.load(cfg['parser']), encoder.model
builder.tokenizer, builder.device = encoder.tokenizer, encoder.model.device
GRAPH_NAMES = {split: f'pearl_hc3_{split}_seed{SEED}' for split in ('train', 'validation')}
GRAPH_PATHS = {split: REPO / 'graph_data' / f'{name}.pkl' for split, name in GRAPH_NAMES.items()}
assert not any(path.exists() for path in GRAPH_PATHS.values()), 'Không ghi đè cache có sẵn.'
assert len(list((REPO / 'graph_data').glob('*.pkl'))) + 2 <= 4, 'Vượt giới hạn 4 graph caches.'
for split in ('train', 'validation'):
    docs = read_jsonl(RUN / 'prepared' / f'{split}_docs.jsonl.gz')
    rows = [json.dumps({'text': d['text'], 'label': d['label']}) for d in docs]
    embeddings, edges, labels = builder.build_graph(rows, return_edge_type=False)
    assert len(embeddings) == len(edges) == len(labels) == len(docs)
    builder.save_pkl(GRAPH_NAMES[split], embeddings, edges, labels)
    del embeddings, edges, labels, docs, rows
builder.nlp = builder.model = builder.tokenizer = builder.device = None
del encoder
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
baseline_args = ['gcn.py', '--dataset', 'hc3_pearl', '--seed', str(SEED),
    '--train_file', GRAPH_NAMES['train'], '--val_file', GRAPH_NAMES['validation'],
    '--epochs', str(2 if SMOKE_TEST else BASELINE_EPOCHS), '--patience', str(PATIENCE),
    '--batch_size', '32', '--save_path', BASELINE, '--no_test']
if DEVICE == 'cpu':
    baseline_args.append('--cpu')
run('baseline_hc3', baseline_args)
assert BASELINE.is_file()
for path in GRAPH_PATHS.values():
    path.unlink()  # Chỉ xóa 2 cache vừa tạo, giữ nguyên mọi baseline có sẵn.


## Train predictor rồi fine-tune PEARL-GCN
Predictor chỉ dùng graph hiện tại. Joint loss: clean BCE + lambda_edge × valid edge BCE cân bằng theo
document. Chỉ HC3 validation chọn checkpoint; test GPT/Insert chưa được parse ở các bước train.


In [ ]:
run('edge_predictor', ['-m', 'pearl.train_edge', '--config', CONFIG])
run('pearl_hc3', ['-m', 'pearl.train_gcn', '--config', CONFIG])
CHECKPOINT = MODEL_DIR / 'pearl/best.pt'
assert CHECKPOINT.is_file()
# Không giữ embeddings sau khi train; evaluator đọc từng text, không tạo graph .pkl test.
for name in ('train_embeddings.pt', 'validation_embeddings.pt'):
    path = RUN / 'prepared' / name
    if path.is_file():
        path.unlink()


## Test sau khi train: GPT3.5-Mixed 0% và HC3 Insert 25%
Chạy đúng 2 điều kiện, không mở các điều kiện khác. Full run: 1,000 mẫu mỗi tập;
smoke: 4 mẫu mỗi tập. Có accuracy, AUROC, F1, recall, confusion matrix và từng xác suất dự đoán.


In [ ]:
REPORT = EVAL_DIR / f'metrics_seed{SEED}.json'
arguments = ['-m', 'pearl.evaluate', '--config', CONFIG, '--output', REPORT]
if SMOKE_TEST:
    arguments += ['--limit', '4']
run('evaluate', arguments)
report = json.loads(REPORT.read_text(encoding='utf-8'))
assert len(report['results']) == 2
for result in report['results']:
    assert result['status'] == 'ok', result
    assert result['metrics']['n'] == (4 if SMOKE_TEST else 1000)
    print(json.dumps({'seed': SEED, 'dataset': result['dataset'], 'condition': result['condition'],
                      **result['metrics']}, indent=2))


In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile
metadata = {'seed': SEED, 'n_seeds': 1, 'branch': BRANCH, 'commit': COMMIT, 'smoke': SMOKE_TEST,
    'baseline_epochs': 2 if SMOKE_TEST else BASELINE_EPOCHS,
    'edge_epochs': cfg['edge_epochs'], 'pearl_epochs': cfg['gcn_epochs'],
    'test_conditions': ['gpt35_mixed_0pct', 'hc3_insert25'],
    'torch': str(torch.__version__), 'spacy': spacy.__version__}
METADATA = RUN / f'run_seed{SEED}.json'
METADATA.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
ENVIRONMENT = RUN / 'pip-freeze.txt'
ENVIRONMENT.write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True), encoding='utf-8')
suffix = '_smoke' if SMOKE_TEST else ''
ZIP_PATH = WORK / f'pearl_lite_hc3_to_gpt35_0pct_hc3_insert25_seed{SEED}{suffix}.zip'
assert not ZIP_PATH.exists(), 'Không ghi đè ZIP run cũ.'
files = [BASELINE, CHECKPOINT, CONFIG, REPORT, METADATA, ENVIRONMENT,
    MODEL_DIR / 'edge/edge.pt', MODEL_DIR / 'edge/metrics.json', MODEL_DIR / 'pearl/training.json',
    RUN / 'prepared/manifest.json', *sorted((RUN / 'logs').glob('*.log'))]
assert all(path.is_file() for path in files)
with ZipFile(ZIP_PATH, 'w', ZIP_DEFLATED) as archive:
    for path in files:
        archive.write(path, path.relative_to(RUN))
with ZipFile(ZIP_PATH) as archive:
    assert archive.testzip() is None
print({'zip': str(ZIP_PATH), 'bytes': ZIP_PATH.stat().st_size, 'seed': SEED, 'n_seeds': 1})
from IPython.display import display, FileLink
display(FileLink(str(ZIP_PATH)))
